# PaperBanana — Colab Quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/llmsresearch/paperbanana/blob/main/notebooks/PaperBanana_Colab_Quickstart.ipynb)

[PaperBanana](https://github.com/llmsresearch/paperbanana) generates publication-quality
methodology diagrams and statistical plots from plain-text descriptions, using a
multi-agent pipeline (Retriever → Planner → Stylist → Visualizer ⇄ Critic).

This notebook runs end-to-end in about 5 minutes:

1. Install PaperBanana from PyPI
2. Set a **Google Gemini API key** (free tier available)
3. Generate a methodology diagram from a tiny inline example
4. *(Optional)* Generate a statistical plot from inline CSV data
5. *(Optional)* Download the full reference set and run an evaluation

**What you need:** a free Gemini API key from
[Google AI Studio](https://aistudio.google.com/app/apikey).
PaperBanana also supports OpenAI, Azure OpenAI, Atlas Cloud, and OpenRouter —
see the [README](https://github.com/llmsresearch/paperbanana#providers) — but Gemini
is the default and has a free tier, so that is what this notebook uses.

## 1. Install

Install PaperBanana from PyPI with the Google Gemini provider extra.

In [ ]:
%pip install -q "paperbanana[google]"

## 2. Set your API key

Get a free key from [Google AI Studio](https://aistudio.google.com/app/apikey) and paste it
below (the prompt hides your input; nothing is stored in the notebook).

> **Using OpenAI instead?** Set `os.environ["OPENAI_API_KEY"]` here and add
> `--vlm-provider openai --image-provider openai_imagen` to the commands below.

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

## 3. Write a tiny methodology example

PaperBanana takes a plain-text description of your method as input. We write a small
example inline so the notebook has no external dependencies — replace it with your
own methodology text later.

In [ ]:
methodology = """\
Our model builds on the Transformer architecture with several key modifications.

Input tokens are embedded through a learned embedding layer and combined with
sinusoidal positional encodings. The result passes through a stack of N=12 encoder
layers, each consisting of multi-head self-attention (8 heads) followed by a
position-wise feed-forward network with GELU activation. Layer normalization is
applied before each sub-layer (Pre-LN), with residual connections around each.

The decoder mirrors this structure but adds a cross-attention layer between the
self-attention and feed-forward sub-layers, attending to the encoder output.
Causal masking prevents the decoder from attending to future positions.

We introduce a sparse attention pattern in the encoder that reduces quadratic
complexity to O(n sqrt(n)): a learned router predicts attention scores for all
positions and selects the top-k positions for each query.

The final decoder output is projected through a linear layer followed by softmax
to produce output token probabilities.
"""

with open("method.txt", "w") as f:
    f.write(methodology)

print(f"Wrote method.txt ({len(methodology)} chars)")

## 4. Generate the diagram

We run a single refinement iteration (`--iterations 1`) to keep this fast and cheap,
and cap spend with `--budget 0.50` (USD) — the pipeline aborts gracefully if the cap
would be exceeded. The default config already uses Gemini for both the VLM and image
generation.

This typically takes 1–3 minutes.

In [ ]:
!paperbanana generate \
  --input method.txt \
  --caption "Overview of our sparse-routing encoder-decoder architecture" \
  --iterations 1 \
  --budget 0.50 \
  --output diagram.png

Display the result inline:

In [ ]:
from IPython.display import Image, display

display(Image("diagram.png"))

## 5. (Optional) Statistical plot from CSV data

`paperbanana plot` renders charts via VLM-generated matplotlib code — **no
image-generation model is involved**, so it is cheap and fast. We write a small
CSV inline.

In [ ]:
csv_data = """\
model,MMLU,GSM8K,HumanEval
Baseline,62.1,48.3,35.4
Ours (dense),68.7,55.2,41.8
Ours (sparse),70.2,58.9,44.1
"""

with open("results.csv", "w") as f:
    f.write(csv_data)

print(csv_data)

In [ ]:
!paperbanana plot \
  --data results.csv \
  --intent "Grouped bar chart comparing model accuracy across three benchmarks" \
  --iterations 1 \
  --budget 0.25 \
  --output plot.png

In [ ]:
from IPython.display import Image, display

display(Image("plot.png"))

## 6. (Optional) Download the full reference set

PaperBanana ships with a small curated set of reference diagrams, which the Retriever
agent uses for in-context planning. For best results you can download the full
**PaperBananaBench** reference set.

> **Note:** this is a **~254 MB** download (cached under `~/.cache/paperbanana/`),
> so skip this cell if you are just exploring.

In [ ]:
!paperbanana data download
!paperbanana data info

## 7. (Optional) Evaluate against a human reference

`paperbanana evaluate` scores a generated diagram against a human-made reference using
a VLM-as-a-Judge on four dimensions: **Faithfulness**, **Readability**, **Conciseness**,
and **Aesthetics**.

Normally you would pass your own paper's original figure as `--reference`. As a demo,
we borrow one image from the reference set downloaded in step 6 (so run that cell first).

In [ ]:
import glob
import os.path

refs = sorted(glob.glob(os.path.expanduser("~/.cache/paperbanana/reference_sets/images/*")))
assert refs, "No reference images found - run the download cell in step 6 first."
reference_image = refs[0]
print(f"Using demo reference: {reference_image}")

In [ ]:
!paperbanana evaluate \
  --generated diagram.png \
  --reference "{reference_image}" \
  --context method.txt \
  --caption "Overview of our sparse-routing encoder-decoder architecture"

## Next steps

- **Better quality:** add `--optimize` (input enrichment) and `--auto` (refine until the
  critic is satisfied) to `paperbanana generate`.
- **Batch mode:** generate many figures from one YAML manifest with `paperbanana batch`.
- **Local web UI:** `pip install 'paperbanana[studio]'` then `paperbanana studio`.
- **MCP server:** use PaperBanana from Claude Code or Cursor via `paperbanana-mcp`.
- **Docker:** `docker build -t paperbanana . && docker run --rm -e GOOGLE_API_KEY paperbanana --help`

Full documentation: [github.com/llmsresearch/paperbanana](https://github.com/llmsresearch/paperbanana)

> PaperBanana is an unofficial, community-driven implementation of
> ["PaperBanana: Automating Academic Illustration for AI Scientists"](https://arxiv.org/abs/2601.23265).